In [1]:
url = "https://www.billboard.com/charts/billboard-200"

In [2]:
!ls

Projet-billboard.ipynb	newscrawler


In [3]:
!pip install scrapy

In [4]:
!scrapy startproject newscrawler

Error: scrapy.cfg already exists in /home/dev/code/6Evaluation/Projet/Projet-billboard/newscrawler


In [5]:
# %load newscrawler/newscrawler/items.py

# Define here the models for your scraped items
#
# See documentation in:
# https://doc.scrapy.org/en/latest/topics/items.html

import scrapy

class ArticleItem(scrapy.Item):
    title = scrapy.Field()
    image = scrapy.Field()
    description = scrapy.Field()

class NewscrawlerItem(scrapy.Item):
    # define the fields for your item here like:
    # name = scrapy.Field()
    pass


In [6]:
# %load newscrawler/newscrawler/spiders/billboard.py
import scrapy
from scrapy import Request
# from items import ArticleItem
class LemondeSpider(scrapy.Spider):
    name = "billboard"
    allowed_domains = ["www.billboard.com/charts/billboard-200"]
    start_urls = ['https://www.billboard.com/charts/billboard-200']

    def parse(self, response):
        title = response.css('title::text').extract_first()
        all_links = {
            name:response.urljoin(url) for name, url in zip(
            response.css("#nav-markup .Nav__item")[3].css("a::text").extract(),
            response.css("#nav-markup .Nav__item")[3].css("a::attr(href)").extract())
        }
        for link in all_links.values():
            yield Request(link, callback=self.parse_category)

    def parse_category(self, response):
        for article in response.css(".river")[0].css(".teaser"):
            title = self.clean_spaces(article.css("h3 ::text").extract_first())
            image = article.css("img::attr(data-src)").extract_first()
            description = article.css("p::text").extract_first()

            yield ArticleItem(
                title=title,
                image=image,
                description=description
            )

    def clean_spaces(self, string):
        if string:
            return " ".join(string.split())


In [7]:
!cd newscrawler/newscrawler/spiders/ && scrapy crawl billboard

2020-01-15 09:48:44 [scrapy.utils.log] INFO: Scrapy 1.3.3 started (bot: newscrawler)
2020-01-15 09:48:44 [scrapy.utils.log] INFO: Overridden settings: {'BOT_NAME': 'newscrawler', 'NEWSPIDER_MODULE': 'newscrawler.spiders', 'ROBOTSTXT_OBEY': True, 'SPIDER_MODULES': ['newscrawler.spiders']}
2020-01-15 09:48:44 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.logstats.LogStats']
2020-01-15 09:48:44 [scrapy.middleware] INFO: Enabled downloader middlewares:
['scrapy.downloadermiddlewares.robotstxt.RobotsTxtMiddleware',
 'scrapy.downloadermiddlewares.httpauth.HttpAuthMiddleware',
 'scrapy.downloadermiddlewares.downloadtimeout.DownloadTimeoutMiddleware',
 'scrapy.downloadermiddlewares.defaultheaders.DefaultHeadersMiddleware',
 'scrapy.downloadermiddlewares.useragent.UserAgentMiddleware',
 'scrapy.downloadermiddlewares.retry.RetryMiddleware',
 'scrapy.downloadermiddlewares.redirect.MetaRefreshM

sh: 0: getcwd() failed: No such file or directory
/bin/sh: 1: cd: getcwd() failed: No such file or directory
